# Add required libraries

To connect to a mysql database we must use a connector. We can add the connector with: ``` uv add mysql-connector-python ```

Not a bad idea to update all of our uv packages at this point. ``` uv sync --upgrade ```
    

# Python module names 

```name=="main":```

Python files are referred to as “modules”. A module can have functions, classes, and variables. When the Python interpreter runs a module, the `__name__` variable will be set as `__main__` if the module being run is the main program. But if the code is being run from another module, then the `__name__` variable will be set to that module’s (file) name.

So:

1. Every Python module has a special variable called **name**. When a module is executed as the main program, the **name** variable is set to the string '**main**'. However, if the module is being imported into another program, the **name** variable is set to the name of the module.
2. The if **name** == '**main**': block allows you to specify code that should only be executed when the module is run as the main program. This block is typically placed at the end of the module, after all the function and class definitions.
3. Any code inside the if **name** == '**main**': block will be executed only if the module is run directly as the main program. If the module is imported into another program, the code inside this block will be skipped.

# Connecting Python to a database

Before we can begin writing our program to interact with the database, we must ensure the database exists and has tables with the appropriate structure. The database to use will be ‘infs3070’ and each student can create their own database table by either using PHPMyAdmin, or by using the SQL script from the command line.

## Option 1 - Connecting within our python program

In [2]:
import mysql.connector

def create_conn():
    conn = mysql.connector.connect(
        host = 'localhost',
        user = 'infs3070',  # More secure would be: user = os.environ.get('3070pass')
        password = 'pydev', # More secure would be: password = os.environ.get('3070pass')
        database = 'INFS3070'

    )

    return conn

# The rest of my Python program would be below here.....

## Option 2 - Separate connection program.

### Selecting (READ) records from a database.

In [10]:
from dbConfig import create_conn

# Create a function to fetch all student records
def get_all_students():
    conn = create_conn()
    cursor = conn.cursor()

    select_query = 'SELECT * FROM STUDENTS;'
    cursor.execute(select_query)

    results = cursor.fetchall()
    print("All student records: ")
    print("---------------------")

    for row in results:
        print(row)

    cursor.close()
    conn.close()


if __name__ == "__main__":
    get_all_students()



All student records: 
---------------------
(1, 'Cook', 'Bob', 'rcook2@uccs.edu')
(2, 'Schmidt', 'John Jacob', 'jj@email.com')
(3, 'Poppins', 'Mary', 'mp@email.com')
(5, 'Smith', 'John', 'js@email.com')
(6, 'Quest', 'Johnny', 'jq@email.com')


### Insert records into the database

In [7]:
from dbConfig import create_conn

# Function to add records
def add_student(last_name, first_name, email_input):
    conn = create_conn()
    cursor = conn.cursor()

    # Store values as placeholders for insertion.
    insert_query = 'INSERT INTO STUDENTS (lastName, firstName, email) VALUES (%s, %s, %s)'
    record = (last_name, first_name, email_input)

    cursor.execute(insert_query, record)
    conn.commit()

    print("Student record added successfully!")
    cursor.close()
    conn.close()

# Adding the actual student data
if __name__ == "__main__":
    last_name = 'user'
    first_name = 'test' 
    email_input = 'testUser@email.com'

    add_student(last_name, first_name, email_input)





Student record added successfully!


### Update records

In [9]:
from dbConfig import create_conn

# Function for updating student record
def update_student(student_id, last_name, first_name, email_update):
    conn = create_conn()
    cursor = conn.cursor()

    update_query = """
        update STUDENTS
        set lastName = %s, firstName = %s, email = %s
        where studentID = %s
        """


    cursor.execute(update_query, (last_name, first_name, email_update, student_id))
    conn.commit()

    print("Student record updated successfully!")
    cursor.close()
    conn.close()


# Add values for update
updated_student_id = 6
updated_last_name = 'Quest'
updated_first_name = 'Johnny'
updated_email = 'jq@email.com'
update_student(updated_student_id, updated_last_name, updated_first_name, updated_email)

    

Student record updated successfully!


### Delete record

In [9]:
from dbConfig import create_conn

# Function to delete record
def delete_student(student_id_input):
    conn = create_conn()
    cursor = conn.cursor()

    delete_query = 'DELETE FROM STUDENTS WHERE studentID = %s'
    cursor.execute(delete_query, (student_id_input,)) # The comma is a strange Python quirk to tell it, "no more fields" 

    conn.commit()

    if cursor.rowcount > 0:
        print("Student record deleted successfully!")
    else:
        print("No record found with the given student ID.")

    cursor.close()
    conn.close()


# Add the record value to delete
student_id = 8
delete_student(student_id)



Student record deleted successfully!


# Connecting to Company database

In [11]:
from companyConnect import get_conn_params
import pandas as pd
from sqlalchemy import create_engine, text

params = get_conn_params()

# Construct SQLAlchemy connect URL
engine = create_engine("mysql+pymysql://{user}:{password}@{host}/{database}".format(**params))

df = pd.read_sql("SELECT count(*) from sales;", engine)

print(df)

   count(*)
0    367213


In [13]:
departments = pd.read_sql("SELECT * FROM departments;", engine)

departments.head(5)

,Dept,DeptDescription
0,1,Candy and Tobacco
1,2,Health and Beauty Aids
2,3,Stationery
3,4,Paper Goods
4,5,Media and Gaming Cafe
